Bohua Liu is responsible for this code

**From the previous data analysis we understand that is unrational to use all 60 categories which cannot guarantee that each cat has sufficient training samples -> we group these 60 cats into 4 main cats**

Another interesting point is that we **removed test dataset**.

Since this is not a data challenge task, we could only account for val set to have idea of the performance. This choice is debatable, but doing so we can increse the sample images in both training and val dataset, which we believed to be very important.


In [ ]:
COCO_JSON   = "TACO/data/annotations.json"   # TACO COCO annotations
IMAGES_ROOT = "TACO/data"                    # TACO images root

# Output YOLO dataset root (DETECTION now)
YOLO_ROOT   = "yolo_4cats"      

# Split ratios
TRAIN_PCT = 0.85
VAL_PCT   = 0.15

# Random seed for reproducibility
SEED = 42

import os, json, shutil, random
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict, Counter

random.seed(SEED)

YOLO_ROOT = Path(YOLO_ROOT)
IMG_DIRS = {
    "train": YOLO_ROOT / "images/train",
    "val":   YOLO_ROOT / "images/val",
}
LABEL_DIRS = {
    "train": YOLO_ROOT / "labels/train",
    "val":   YOLO_ROOT / "labels/val",
}
for d in list(IMG_DIRS.values()) + list(LABEL_DIRS.values()):
    d.mkdir(parents=True, exist_ok=True)

# Load COCO JSON
with open(COCO_JSON, "r") as f:
    coco = json.load(f)

images      = {im["id"]: im for im in coco["images"]}
annotations = coco["annotations"]
categories  = coco["categories"]

# CLASS MAPPING (same as your RetinaNet-style script)
#    Map many TACO fine-grained classes into 4 super-classes

CLASS_MAP = {
    # plastic
    "clear plastic bottle": "plastic",
    "crisp packet": "plastic",
    "disposable food container": "plastic",
    "disposable plastic cup": "plastic",
    "foam cup": "plastic",
    "foam food container": "plastic",
    "garbage bag": "plastic",
    "other plastic": "plastic",
    "other plastic bottle": "plastic",
    "other plastic container": "plastic",
    "other plastic cup": "plastic",
    "other plastic wrapper": "plastic",
    "plastic bottle cap": "plastic",
    "plastic film": "plastic",
    "plastic glooves": "plastic",
    "plastic lid": "plastic",
    "plastic straw": "plastic",
    "plastic utensils": "plastic",
    "polypropylene bag": "plastic",
    "single-use carrier bag": "plastic",
    "six pack rings": "plastic",
    "spread tub": "plastic",
    "squeezable tube": "plastic",
    "styrofoam piece": "plastic",
    "tupperware": "plastic",

    # glass
    "broken glass": "glass",
    "glass bottle": "glass",
    "glass cup": "glass",
    "glass jar": "glass",

    # paper
    "corrugated carton": "paper",
    "drink carton": "paper",
    "egg carton": "paper",
    "magazine paper": "paper",
    "meal carton": "paper",
    "normal paper": "paper",
    "other carton": "paper",
    "paper bag": "paper",
    "paper cup": "paper",
    "paper straw": "paper",
    "pizza box": "paper",
    "tissues": "paper",
    "toilet tube": "paper",
    "wrapping paper": "paper",

    # unsorted (metal, food, mixed, unknown)
    "aerosol": "unsorted",
    "aluminium blister pack": "unsorted",
    "aluminium foil": "unsorted",
    "battery": "unsorted",
    "carded blister pack": "unsorted",
    "cigarette": "unsorted",
    "drink can": "unsorted",
    "food can": "unsorted",
    "food waste": "unsorted",
    "metal bottle cap": "unsorted",
    "metal lid": "unsorted",
    "plastified paper bag": "unsorted",  # composite
    "pop tab": "unsorted",
    "rope & strings": "unsorted",
    "scrap metal": "unsorted",
    "shoe": "unsorted",
    "unlabeled litter": "unsorted",
}

# Target IDs (1..4) for convenience in dominant-class logic
TARGET_IDS = {"plastic": 1, "glass": 2, "paper": 3, "unsorted": 4}

# YOLO class list and order (0..3)
names = ["plastic", "glass", "paper", "unsorted"]
name_to_yolo_idx = {n: i for i, n in enumerate(names)}

# Helper map: COCO category id -> original TACO category name
cat_id_to_name = {c["id"]: c["name"] for c in categories}

# Build img_storage for splitting (dominant class logic)

img_storage = {}  # {img_id: {'file_name': str, 'anns': []}}

# Initialize all images
for img in coco["images"]:
    img_storage[img["id"]] = {"file_name": img["file_name"], "anns": []}

# Fill annotations with mapped 1..4 target IDs (for splitting & stats)
for ann in annotations:
    img_id = ann["image_id"]
    if img_id not in img_storage:
        continue

    raw_name = cat_id_to_name.get(ann["category_id"], "")
    old_name = raw_name.strip().lower()

    super_cat = CLASS_MAP.get(old_name, "unsorted")
    new_id = TARGET_IDS[super_cat]  # 1..4

    img_storage[img_id]["anns"].append({
        "bbox": ann["bbox"],
        "category_id": new_id,
    })

# Stratify by dominant class and create train/val split
buckets = defaultdict(list)         # dominant class (1..4) -> list of img_ids
empty_imgs = []                     # images with no annotations
dominant_class_counts = Counter()   # count images dominated by each class (1..4)

for img_id, data in img_storage.items():
    if not data["anns"]:
        empty_imgs.append(img_id)
        continue

    cats = [a["category_id"] for a in data["anns"]]  # 1..4
    dominant_cat = Counter(cats).most_common(1)[0][0]
    buckets[dominant_cat].append(img_id)
    dominant_class_counts[dominant_cat] += 1

print("\n--- Dominant class distribution BEFORE split ---")
for cid in sorted(dominant_class_counts.keys()):
    cname = names[cid - 1]   # 1..4 -> 0..3
    print(f"{cname:10s} : {dominant_class_counts[cid]} images")
print(f"{'empty/no-label':10s} : {len(empty_imgs)} images")

train_ids, val_ids = [], []

for cat, ids in buckets.items():
    random.shuffle(ids)
    n_total = len(ids)
    n_train = int(round(n_total * TRAIN_PCT))
    n_val   = n_total - n_train

    train_ids.extend(ids[:n_train])
    val_ids.extend(ids[n_train:n_train + n_val])

# Handle empty images similarly
random.shuffle(empty_imgs)
n_total = len(empty_imgs)
n_train = int(round(n_total * TRAIN_PCT))
n_val   = n_total - n_train

train_ids.extend(empty_imgs[:n_train])
val_ids.extend(empty_imgs[n_train:n_train + n_val])

splits = {
    "train": set(train_ids),
    "val":   set(val_ids),
}

def count_classes(id_list):
    c = Counter()
    for img_id in id_list:
        data = img_storage[img_id]
        anns = data["anns"]
        if not anns:
            c["empty"] += 1
        else:
            cats = [a["category_id"] for a in anns]  # 1..4
            dominant_cat = Counter(cats).most_common(1)[0][0]
            cname = names[dominant_cat - 1]  # convert 1..4 -> 0..3 name
            c[cname] += 1
    return c

print("\n--- Train split per-class (by dominant class) ---")
train_counts = count_classes(train_ids)
for k, v in train_counts.items():
    print(f"{k:10s}: {v}")

print("\n--- Val split per-class (by dominant class) ---")
val_counts = count_classes(val_ids)
for k, v in val_counts.items():
    print(f"{k:10s}: {v}")

# Build anns_by_image (original COCO anns, used for bboxes now)
anns_by_image = defaultdict(list)
for a in annotations:
    anns_by_image[a["image_id"]].append(a)

# Helper: COCO bbox -> YOLO DET rows (4-class mapping)
#    YOLO det format:
#        <class_idx> xc yc w h   (all normalized [0,1])
def coco_bbox_to_yolo_row(ann, width, height):   # <<< CHANGED
    # Skip crowd if you like
    if ann.get("iscrowd", 0) == 1:
        return None

    # Map COCO category -> TACO name -> 4-class super_cat -> YOLO idx (0..3)
    raw_name = cat_id_to_name.get(ann["category_id"], "")
    old_name = raw_name.strip().lower()
    super_cat = CLASS_MAP.get(old_name, "unsorted")
    yolo_idx = name_to_yolo_idx[super_cat]

    x, y, w, h = ann["bbox"]  # COCO: top-left (x,y), width, height in pixels

    if w <= 0 or h <= 0:
        return None

    xc = (x + w / 2.0) / width
    yc = (y + h / 2.0) / height
    wn = w / width
    hn = h / height

    # Clamp to [0,1] just in case
    xc = min(max(xc, 0.0), 1.0)
    yc = min(max(yc, 0.0), 1.0)
    wn = min(max(wn, 0.0), 1.0)
    hn = min(max(hn, 0.0), 1.0)

    return f"{yolo_idx} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}"

# Convert one image: copy file + write YOLO DET label   <<< CHANGED

def convert_one_image(img_id, split):  # split is "train" or "val"
    im = images[img_id]
    im_path = im["file_name"]           # e.g. "batch_1/000006.jpg"
    src = Path(IMAGES_ROOT) / im_path   # e.g. TACO/data/batch_1/000006.jpg

    if not src.exists():
        return False

    # destination image path
    img_saving_dest = IMG_DIRS[split] / f"{img_id}_{src.name}"  # use id to avoid clashes
    shutil.copy2(src, img_saving_dest)

    # YOLO det label file
    label_path = LABEL_DIRS[split] / (img_saving_dest.stem + ".txt")
    W, H = im["width"], im["height"]

    rows = []
    for ann in anns_by_image.get(img_id, []):
        row = coco_bbox_to_yolo_row(ann, W, H)
        if row is not None:
            rows.append(row)

    # Even if no boxes, we still create the file (possibly empty),
    # so YOLO knows the image exists.
    with open(label_path, "w") as f:
        f.write("\n".join(rows))

    return True

# Run conversion for train/val

counts = {"train": 0, "val": 0}
for split, idset in splits.items():
    print(f"\nConverting {split.upper()} set ({len(idset)} images)...")
    for img_id in tqdm(idset, desc=f"{split.upper()} progress", unit="img"):
        if convert_one_image(img_id, split):
            counts[split] += 1

print("\nConverted images:", counts)
print("Class count:", len(names))
print("Classes:", names)
print(f"\nYOLO detection dataset saved to: {YOLO_ROOT}")



--- Dominant class distribution BEFORE split ---
plastic    : 889 images
glass      : 52 images
paper      : 175 images
unsorted   : 384 images
empty/no-label : 0 images

--- Train split per-class (by dominant class) ---
glass     : 44
paper     : 149
plastic   : 756
unsorted  : 326

--- Val split per-class (by dominant class) ---
glass     : 8
paper     : 26
plastic   : 133
unsorted  : 58

Converting TRAIN set (1275 images)...


TRAIN progress: 100%|██████████| 1275/1275 [00:05<00:00, 239.49img/s]



Converting VAL set (225 images)...


VAL progress: 100%|██████████| 225/225 [00:00<00:00, 243.36img/s]


Converted images: {'train': 1275, 'val': 225}
Class count: 4
Classes: ['plastic', 'glass', 'paper', 'unsorted']

YOLO detection dataset saved to: yolo_4cats


Solve JPG uppercase (some jpg file are written i uppercase JPG)

In [ ]:
import os

# CHANGE THIS
dataset_path = "yolo_4cats"  # Path to the dataset root

image_dirs = [
    os.path.join(dataset_path, "images/train"),
    os.path.join(dataset_path, "images/val")
]

for folder in image_dirs:
    for file in os.listdir(folder):
        if file.endswith(".JPG"):
            old_path = os.path.join(folder, file)
            new_path = os.path.join(folder, file[:-4] + ".jpg")

            print(f"Renaming: {old_path}  →  {new_path}")
            os.rename(old_path, new_path)

print("\nDone! All .JPG converted to .jpg")


Renaming: yolo_4cats/images/train/1254_000093.JPG  →  yolo_4cats/images/train/1254_000093.jpg
Renaming: yolo_4cats/images/train/1218_000054.JPG  →  yolo_4cats/images/train/1218_000054.jpg
Renaming: yolo_4cats/images/train/1187_000016.JPG  →  yolo_4cats/images/train/1187_000016.jpg
Renaming: yolo_4cats/images/train/830_IMG_4893.JPG  →  yolo_4cats/images/train/830_IMG_4893.jpg
Renaming: yolo_4cats/images/train/1135_000042.JPG  →  yolo_4cats/images/train/1135_000042.jpg
Renaming: yolo_4cats/images/train/772_000095.JPG  →  yolo_4cats/images/train/772_000095.jpg
Renaming: yolo_4cats/images/train/1124_000054.JPG  →  yolo_4cats/images/train/1124_000054.jpg
Renaming: yolo_4cats/images/train/928_000058.JPG  →  yolo_4cats/images/train/928_000058.jpg
Renaming: yolo_4cats/images/train/1007_000108.JPG  →  yolo_4cats/images/train/1007_000108.jpg
Renaming: yolo_4cats/images/train/47_000065.JPG  →  yolo_4cats/images/train/47_000065.jpg
Renaming: yolo_4cats/images/train/1104_000075.JPG  →  yolo_4cats/i

Create yaml file

In [ ]:
taco_yaml = YOLO_ROOT / "taco.yaml"
abs_root  = Path("/Users/yw/Desktop/final project files") # Modify it if you have different one

with open(taco_yaml, "w") as f:
    f.write(f"""

# Auto-generated YOLO dataset config for TACO
path: {abs_root}
train: images/train
val: images/val
names:

""")
    for i, n in enumerate(names):
        f.write(f"  {i}: {n}\n")

print("\nWrote:", taco_yaml)
print(taco_yaml.read_text()[:400], " ...")


Wrote: yolo_4cats/taco.yaml


# Auto-generated YOLO dataset config for TACO
path: /Users/yw/Desktop/final project files
train: images/train
val: images/val
names:

  0: plastic
  1: glass
  2: paper
  3: unsorted
  ...


showing some images